# Intra-rater coding-drift diagnostic

**Does Where You Are Shape What You Get?**  
MSc Data Science Dissertation — 2026

This notebook compares two coding occasions by the same researcher for a 40-response subset of the 160-response fresh hold-out sample. One uncodeable repeat response is excluded, leaving 39 matched pairs.

## Interpretation boundary

This exercise does **not** establish acceptable intra-rater reliability because the coding instrument was not invariant across occasions. Round 1 used a 0–2 actionability scale, whereas Round 2 used a 0–4 scale, and auditing found shifts in the interpretation of several binary definitions. Therefore:

- direct ordinal kappa for actionability is not calculated;
- the binarised actionability comparison is a secondary cross-scale diagnostic only;
- neither round is treated as a gold standard;
- disagreements are reported neutrally as 0→1 and 1→0 shifts;
- the results diagnose coding drift and do not validate the automated pipeline;
- a new blinded recoding with the original frozen codebook and 0–2 scale is required for a valid intra-rater estimate.

The corrected interpretation is consistent with the submission-ready reliability supplement.

In [6]:
from google.colab import files

# This will prompt you to select files from your local machine
uploaded = files.upload()

Saving Fresh_Holdout_REPEAT_CODING_LABELED (1).xlsx to Fresh_Holdout_REPEAT_CODING_LABELED (1).xlsx


In [10]:
from google.colab import files
print("Please upload 'Fresh_Holdout_Validation_LABELLED.xlsx' below:")
uploaded_r1 = files.upload()

Please upload 'Fresh_Holdout_Validation_LABELLED.xlsx' below:


Saving Fresh_Holdout_Validation_LABELLED.xlsx to Fresh_Holdout_Validation_LABELLED.xlsx


In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from sklearn.metrics import cohen_kappa_score, confusion_matrix

R1_PATH = "Fresh_Holdout_Validation_LABELLED.xlsx"
R2_PATH = "Fresh_Holdout_REPEAT_CODING_LABELED (1).xlsx"
SHEET_NAME = "Label Here"
EXCLUDED_REPEAT_ID = "R001"

BINARY_COLS = [
    "coping_step",
    "professional_help",
    "social_support",
    "crisis_action",
    "follow_up",
    "surface_localisation",
    "verified_localisation",
]

ACTION_THRESH_R1 = 2
ACTION_THRESH_R2 = 3

def load_sheet(path, sheet=SHEET_NAME):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet]
    rows = list(ws.iter_rows(values_only=True))
    headers = [str(v) if v is not None else f"col_{i}" for i, v in enumerate(rows[0])]
    df = pd.DataFrame(rows[1:], columns=headers)
    wb.close()
    return df[df[headers[0]].notna()].copy()

def kappa_band(k):
    if pd.isna(k): return "not estimable"
    if k < 0: return "negative"
    if k < .20: return "slight"
    if k < .40: return "fair"
    if k < .60: return "moderate"
    if k < .80: return "substantial"
    return "near-perfect"

def symmetric_binary_summary(a, b):
    a = pd.Series(a).astype(int).reset_index(drop=True)
    b = pd.Series(b).astype(int).reset_index(drop=True)
    cm = confusion_matrix(a, b, labels=[0, 1])
    n00, shift_0_to_1, shift_1_to_0, n11 = cm.ravel()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kappa = cohen_kappa_score(a, b)
    return {
        "n": len(a),
        "kappa": float(kappa),
        "kappa_band": kappa_band(kappa),
        "agreement_pct": float((a == b).mean() * 100),
        "round1_positive_pct": float(a.mean() * 100),
        "round2_positive_pct": float(b.mean() * 100),
        "agree_0_0": int(n00),
        "agree_1_1": int(n11),
        "shift_0_to_1": int(shift_0_to_1),
        "shift_1_to_0": int(shift_1_to_0),
    }

for path in [R1_PATH, R2_PATH]:
    if not os.path.exists(path):
        sys.exit(f"Missing input file: {path}")

r1 = load_sheet(R1_PATH)
r2 = load_sheet(R2_PATH)

required_r1 = {"sample_id", "actionability_overall", *BINARY_COLS}
required_r2 = {"sample_id", "repeat_id", "actionability_overall", *BINARY_COLS}
for label, df, required in [("Round 1", r1, required_r1), ("Round 2", r2, required_r2)]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{label} missing columns: {sorted(missing)}")
    if df["sample_id"].astype(str).str.strip().duplicated().any():
        raise ValueError(f"{label} contains duplicate sample_id values")

r2 = r2[r2["repeat_id"].astype(str).str.strip() != EXCLUDED_REPEAT_ID].copy()
for df in [r1, r2]:
    df["sample_id"] = df["sample_id"].astype(str).str.strip()
    for col in ["actionability_overall", *BINARY_COLS]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

merged = r2[["sample_id", "actionability_overall", *BINARY_COLS]].merge(
    r1[["sample_id", "actionability_overall", *BINARY_COLS]],
    on="sample_id",
    suffixes=("_r2", "_r1"),
    validate="one_to_one",
)

print(f"Matched pairs: {len(merged)}")
print("Round 1 actionability range:", tuple(merged["actionability_overall_r1"].agg(["min", "max"])))
print("Round 2 actionability range:", tuple(merged["actionability_overall_r2"].agg(["min", "max"])))
print("Direct ordinal actionability kappa: NOT CALCULATED (scale mismatch)")

results = []
for col in BINARY_COLS:
    valid = merged[[f"{col}_r1", f"{col}_r2"]].dropna()
    if not valid.isin([0, 1]).all().all():
        raise ValueError(f"{col} contains non-binary values")
    result = {
        "measure": col,
        "analysis_status": "coding-drift diagnostic",
        **symmetric_binary_summary(valid.iloc[:, 0], valid.iloc[:, 1]),
    }
    results.append(result)

valid_a = merged[["actionability_overall_r1", "actionability_overall_r2"]].dropna()
a1 = (valid_a["actionability_overall_r1"] >= ACTION_THRESH_R1).astype(int)
a2 = (valid_a["actionability_overall_r2"] >= ACTION_THRESH_R2).astype(int)
results.append({
    "measure": "actionability_overall_binarised_cross_scale",
    "analysis_status": "secondary diagnostic only; non-equivalent original scales",
    **symmetric_binary_summary(a1, a2),
})

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("intra_rater_coding_drift_diagnostic.csv", index=False)

print("""
INTERPRETATION
The two occasions did not use an invariant coding instrument. The observed
statistics therefore diagnose coding drift; they do not confirm acceptable
intra-rater reliability, validate an automated classifier, or establish a
performance ceiling. A blinded codebook-aligned recoding on the original
0–2 actionability scale is required before a standard reliability claim can
be made. The cross-scale actionability result must not be used as a substitute.
""")

## Dissertation placement

Use only a concise method statement in Chapter 3 and a short diagnostic result in Chapter 4. Interpret the episode in Chapter 5 as a limitation and methodological finding. Keep the full table and corrective protocol in the appendix/supplement. Do not use these statistics to reinstate H1 or H2 as confirmatory results, to strengthen the exploratory crisis-contact finding, or to describe Round 1 as proven ground truth.